In [1]:
import os.path

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', "no-latex"])

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

In [3]:
ITL3_region = gpd.read_file(r"./results/intermediate/ITL3_region.gpkg")
substations = gpd.read_file(r"./results/intermediate/substations.gpkg")

In [4]:
interested_locations = {
    "London": {},
    # "TLI3": {},
    # "TLI4": {},
    # "TLI5": {},
    # "TLI6": {},
    # "TLI7": {},
    "TLH1": {},
    "TLH2": {},
    "TLH3": {},
    "TLJ1":{},
    "TLF1":{},
    "TLF2":{},
    "TLC1":{},
    "TLC2":{},
    "TLD3":{},
    "TLD4":{},
    "TLD6":{},
    "TLG1":{},
    "TLG2":{},
    "TLE3":{},
    "TLE4":{},
}

In [5]:
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region.reset_index(),
        "substation": interested_substations.reset_index()
    }

# Grid Points

In [6]:
from SpatialAllocation.utils.GenerateGrid import generate_grid
from shapely.ops import unary_union
import pickle

C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [7]:
def get_color(category):
    if category == 'industrial':
        return 'red'
    elif category == 'residential':
        return kitblue
    elif category == 'commercial':
        return 'yellow'
    elif category == 'agricultural':
        return kitgreen
    else:
        return grey30

In [8]:
def calculate_step_size(target_points, regions):
    """
    计算合适的网格步长
    :param target_points: 目标点数
    :param regions: 地区的GeoDataFrame
    :return: 合适的网格步长
    """
    regions = regions.to_crs("EPSG:3857")
    boundary_polygon_m = unary_union(regions["geometry"])
    minx, miny, maxx, maxy = boundary_polygon_m.bounds
    box_area = (maxx - minx) * (maxy - miny)
    polygon_area = boundary_polygon_m.area
    area_ratio = polygon_area / box_area
    adjusted_target = target_points / area_ratio
    ideal_step_size = np.sqrt(box_area / adjusted_target)
    step_size_m = np.floor(ideal_step_size / 10) * 10
    return max(10, step_size_m)

In [9]:
for key in interested_locations.keys():
    grid_gdf_path = f"./results/intermediate/GridPoint/{key}_grid_points.pickle"
    print("Processing interested location:", key)
    if os.path.exists(grid_gdf_path):
        with open(grid_gdf_path, "rb") as f:
            grid_gdf, step_size_m = pickle.load(f)
            print(f"step_size_m for {key} is {step_size_m} m")
    else:
        step_size_m = calculate_step_size(target_points=200000, regions=interested_locations[key]["region"].copy())
        # step_size_m = 100
        print(f"Calculated step_size_m for {key} is {step_size_m} m")
        grid_gdf, numerical_col_names, categorical_col_members = generate_grid(interested_locations[key]["region"].copy(), step_size_m=step_size_m, with_tags={"landuse":True})
        print(grid_gdf.shape)

        grid_gdf['color'] = grid_gdf['landuse'].apply(get_color)
        with open(grid_gdf_path, "wb") as f:
            pickle.dump([grid_gdf, step_size_m], f)
    interested_locations[key]["grid"] = grid_gdf
    print(f"grid_gdf:{grid_gdf.shape}")
    interested_locations[key]["step_size_m"] = step_size_m

Processing interested location: London
step_size_m for London is 280.0 m
grid_gdf:(51710, 26)
Processing interested location: TLH1
step_size_m for TLH1 is 820.0 m
grid_gdf:(50322, 91)
Processing interested location: TLH2
step_size_m for TLH2 is 380.0 m
grid_gdf:(52357, 60)
Processing interested location: TLH3
step_size_m for TLH3 is 430.0 m
grid_gdf:(51851, 63)
Processing interested location: TLJ1
step_size_m for TLJ1 is 540.0 m
grid_gdf:(51267, 67)
Processing interested location: TLF1
step_size_m for TLF1 is 510.0 m
grid_gdf:(51089, 77)
Processing interested location: TLF2
step_size_m for TLF2 is 510.0 m
grid_gdf:(50960, 65)
Processing interested location: TLC1
step_size_m for TLC1 is 420.0 m
grid_gdf:(51199, 51)
Processing interested location: TLC2
step_size_m for TLC2 is 580.0 m
grid_gdf:(50807, 54)
Processing interested location: TLD3
step_size_m for TLD3 is 260.0 m
grid_gdf:(53327, 55)
Processing interested location: TLD4
step_size_m for TLD4 is 410.0 m
grid_gdf:(52316, 67)
Proces

In [10]:
grid_gdf

,geometry,index_right0,index,ITL3,population,ITL2,industrial_gva,commercial_gva,agricultural_gva,others_gva,...,lu_residential_prop,lu_retail_prop,lu_shrubs_prop,lu_spoil_prop,lu_sport_prop,lu_storage_prop,lu_traffic_island_prop,lu_village_green_prop,lu_vineyard_prop,color
0,POINT (-1.84343 53.52155),2,32,TLE44,650768.0,TLE4,0.009548,0.005730,0.001880,0.005669,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
1,POINT (-1.84038 53.52155),2,32,TLE44,650768.0,TLE4,0.009548,0.005730,0.001880,0.005669,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
2,POINT (-1.83732 53.52155),2,32,TLE44,650768.0,TLE4,0.009548,0.005730,0.001880,0.005669,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
3,POINT (-1.83427 53.52155),2,32,TLE44,650768.0,TLE4,0.009548,0.005730,0.001880,0.005669,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
4,POINT (-1.82205 53.52155),2,32,TLE44,650768.0,TLE4,0.009548,0.005730,0.001880,0.005669,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50117,POINT (-1.88619 53.9605),0,30,TLE41,560194.0,TLE4,0.006250,0.004572,0.000585,0.005376,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
50118,POINT (-1.88314 53.9605),0,30,TLE41,560194.0,TLE4,0.006250,0.004572,0.000585,0.005376,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#b3b3b3ff
50119,POINT (-1.88008 53.9605),0,30,TLE41,560194.0,TLE4,0.006250,0.004572,0.000585,0.005376,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#009682
50120,POINT (-1.88314 53.96229),0,30,TLE41,560194.0,TLE4,0.006250,0.004572,0.000585,0.005376,...,0.041225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,#009682


In [11]:
# for i, key in enumerate(interested_locations.keys()):
#     fig, ax = plt.subplots(1, 1, figsize=(3.5, 2.5), dpi=300)  # 双栏宽度
#     interested_locations[key]["grid"].plot(ax=ax, color=interested_locations[key]["grid"]["color"], markersize=0.2, aspect='equal', rasterized=True)
#     ax.tick_params(axis='x', labelsize=6)  # 调整x轴刻度标签的大小
#     ax.tick_params(axis='y', labelsize=6)
#     ax.set_xlabel(r'Longitude [$^\circ$E]', fontsize=8)
#     ax.set_ylabel(r'Latitude [$^\circ$N]', fontsize=8)
#     # ax.set_title('Original Surfaces and Points', fontsize=8)
#     # 添加图例
#     legend_elements = [
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=6, label='Industrial'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=kitblue, markersize=6, label='Residential'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow', markersize=6, label='Commercial'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=kitgreen, markersize=6, label='Agricultural'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=6, label='Transportation'),
#         plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=grey30, markersize=6, label='Other')
#     ]
#
#     # 将图例放在 x 轴下方
#     ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.2),
#               ncol=3, fontsize=6)
#
#     plt.tight_layout()
#     # plt.savefig(f'./results/pictures/Figure_interested_region_{key}_with_landuse.pdf', dpi=300, bbox_inches='tight')
#     plt.show()

# Voronoi Diagramm

In [12]:

import os
from SpatialAllocation.voronoi.clustering import do_clustering
from SpatialAllocation.utils import assign_color

In [13]:
from SpatialAllocation.voronoi.core.SimpleVoronoi import simple_voronoi

In [14]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/voronoi_{interested_location}_with_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    substations_key["cluster_label"] = range(len(substations_key))

    if os.path.exists(path):
        print(f"Voronoi grid for {interested_location} with subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} voronoi with subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns={"ITL3": "ITL3"})
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["voronoi_with_ITL3"] = [voronoi_gdf, substations_key]
    print("#" * 50)

Voronoi grid for London with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLH1 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLH2 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLH3 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLJ1 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLF1 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLF2 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLC1 with subcolumn already exists, skipping...
##################################################
Voronoi grid for TLC2 with subcolumn already exists, skipping...
#####

In [15]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/clustered_voronoi_{interested_location}_with_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    coords = np.column_stack((substations_key.geometry.x, substations_key.geometry.y))
    cluster_gdf, centroid_gdf = do_clustering(coords, "hdbscan", min_cluster_size=2)
    cluster_gdf = assign_color(cluster_gdf)
    substations_key[["cluster_label", "color"]] = cluster_gdf[["cluster_label", "color"]]

    if os.path.exists(path):
        print(f"Clustered voronoi grid for {interested_location} with subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} clustered voronoi with subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns={"ITL3": "ITL3"})
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["clustered_voronoi_with_ITL3"] = [voronoi_gdf, substations_key]
    print("#" * 50)

Clustered voronoi grid for London with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLH1 with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLH2 with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLH3 with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLJ1 with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLF1 with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLF2 with subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLC1 with subcolumn already exists, skipping...
#########################################

In [16]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/voronoi_{interested_location}_without_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    substations_key["cluster_label"] = range(len(substations_key))

    if os.path.exists(path):
        print(f"Voronoi grid for {interested_location} without subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} voronoi without subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns=None)
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["voronoi"] = [voronoi_gdf, substations_key]
    print("#" * 50)

Voronoi grid for London without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLH1 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLH2 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLH3 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLJ1 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLF1 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLF2 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLC1 without subcolumn already exists, skipping...
##################################################
Voronoi grid for TLC2 without subcolumn alread

In [17]:
for interested_location in interested_locations.keys():
    path = f"./results/intermediate/Voronoi/clustered_voronoi_{interested_location}_without_subcolumn.gpkg"

    grid_gdf = interested_locations[interested_location]["grid"].copy()
    substations_key = interested_locations[interested_location]["substation"].copy()
    coords = np.column_stack((substations_key.geometry.x, substations_key.geometry.y))
    cluster_gdf, centroid_gdf = do_clustering(coords, "hdbscan", min_cluster_size=2)
    cluster_gdf = assign_color(cluster_gdf)
    substations_key[["cluster_label", "color"]] = cluster_gdf[["cluster_label", "color"]]

    if os.path.exists(path):
        print(f"Clustered voronoi grid for {interested_location} without subcolumn already exists, skipping...")
        voronoi_gdf = gpd.read_file(path)
    else:
        print(f"Processing {interested_location} clustered voronoi without subcolumn")
        voronoi_gdf = simple_voronoi(substations_key, grid_gdf, sub_columns=None)
        voronoi_gdf.to_file(path, driver='GPKG')

    interested_locations[interested_location]["clustered_voronoi"] = [voronoi_gdf, substations_key]
    print("#" * 50)

Clustered voronoi grid for London without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLH1 without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLH2 without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLH3 without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLJ1 without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLF1 without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLF2 without subcolumn already exists, skipping...
##################################################
Clustered voronoi grid for TLC1 without subcolumn already exists, skipping...
#################

In [18]:
import pickle
import os
from ClusterBasedVoronoi.voronoi import build_model
from ClusterBasedVoronoi.clustering import do_clustering
from ClusterBasedVoronoi.utils import assign_color

for key in interested_locations.keys():
    path = f"./results/intermediate/Pyomo/hdbscan_voronoi_result_british_{key}.pkl"
    substations_key = interested_locations[key]["substation"].copy()
    coords = np.column_stack((substations_key.geometry.x, substations_key.geometry.y))
    cluster_gdf, centroid_gdf = do_clustering(coords, "hdbscan", min_cluster_size=2)
    cluster_gdf = assign_color(cluster_gdf)
    substations_key[["cluster_label", "color"]] = cluster_gdf[["cluster_label", "color"]]
    if os.path.exists(path):
        with open(path, "rb") as file:
            voronoi_gdf = pickle.load(file)
    else:
        model, voronoi_gdf, grid_points = build_model(interested_locations[key]["region"], substations_key, interested_locations[key]["step_size_m"], weight="equal", method="civd")
        with open(path, "wb") as file:
            pickle.dump(voronoi_gdf, file)

    interested_locations[key]["cluster_voronoi_hdbscan"] = [voronoi_gdf, substations_key]

In [19]:
# 首先需要对数据进行适配，因为原来的代码结构有所不同
for key in interested_locations.keys():
    regions = interested_locations[key]["region"].copy()

    # 计算residential需求（基于人口）
    # total_england_population = ITL3_region.population.sum()
    # regions["residential_percent"] = regions["population"] / total_england_population  # 28.79%为residential能耗占比

    # 计算各行业需求（基于GVA数据，这些数据在ITL3_region中已经是比例）
    # regions["industrial_percent"] = regions["industrial_gva"]
    # regions["commercial_percent"] = regions["commercial_gva"]
    # regions["agricultural_percent"] = regions["agricultural_gva"]
    # regions["others_percent"] = regions["others_gva"]

    # 归一化
    total_percent = (regions["residential_percent"] + regions["industrial_percent"] +
                    regions["commercial_percent"] + regions["agricultural_percent"] +
                    regions["others_percent"])


    regions["residential_percent"] = regions["residential_percent"] / total_percent
    regions["industrial_percent"] = regions["industrial_percent"] / total_percent
    regions["commercial_percent"] = regions["commercial_percent"] / total_percent
    regions["agricultural_percent"] = regions["agricultural_percent"] / total_percent
    regions["others_percent"] = regions["others_percent"] / total_percent

    # 计算各类型的需求
    regions["residential_demand"] = regions["Demand (MVA)"] * regions["residential_percent"]
    regions["industrial_demand"] = regions["Demand (MVA)"] * regions["industrial_percent"]
    regions["commercial_demand"] = regions["Demand (MVA)"] * regions["commercial_percent"]
    regions["agricultural_demand"] = regions["Demand (MVA)"] * regions["agricultural_percent"]
    regions["others_demand"] = regions["Demand (MVA)"] * regions["others_percent"]

    interested_locations[key]["regions_all"] = regions

In [20]:
regions

,index,ITL3,population,ITL2,residential_percent,industrial_gva,commercial_gva,agricultural_gva,others_gva,industrial_percent,...,area,area_percent,Demand (MVA),Firm Capacity (MVA),geometry,residential_demand,industrial_demand,commercial_demand,agricultural_demand,others_demand
0,30,TLE41,560194.0,TLE4,0.364704,2494,4630,14,4272,0.236601,...,1.051356e+09,0.181442,386.696680,704.40,"MULTIPOLYGON (((-1.8813 53.96308, -1.88041 53....",141.029756,91.492869,66.919922,8.563548,78.690586
1,31,TLE42,829413.0,TLE4,0.242219,4495,14583,68,12811,0.191287,...,1.581531e+09,0.272939,679.760786,1230.66,"MULTIPOLYGON (((-1.34056 53.94487, -1.34074 53...",164.650746,130.029250,166.204155,32.798591,186.078043
2,32,TLE44,650768.0,TLE4,0.328990,3810,5803,45,4505,0.280673,...,2.198437e+09,0.379404,532.428874,895.21,"MULTIPOLYGON (((-1.70293 53.76406, -1.70256 53...",175.163735,149.438149,89.675274,29.429570,88.722146
3,33,TLE45,361786.0,TLE4,0.286720,2662,3509,38,2984,0.307420,...,9.631190e+08,0.166214,329.545098,514.30,"MULTIPOLYGON (((-1.29593 53.73947, -1.29579 53...",94.487009,101.308735,52.614548,24.113338,57.021468


In [21]:
# 为每个grid点分配需求
for key in interested_locations.keys():
    grid_gdf_path = f"./results/intermediate/GridPoint/{key}_grid_points.pickle"
    with open(grid_gdf_path, "rb") as f:
        grid_gdf, step_size_m = pickle.load(f)
    regions = interested_locations[key]["regions_all"].copy()

    # --- 准备工作：空间连接 ---
    # 需要根据grid点所在的ITL3区域来分配需求
    # 首先进行空间连接，找出每个grid点属于哪个ITL3区域
    # 注意：在regions中选择ITL3列，以确保它能被连接到grid_gdf上
    grid_with_regions = grid_gdf.copy()

    # =================================================================================
    # 方案一：按【landuse类型】在各自区域内均分需求 (您的原始逻辑)
    # =================================================================================

    # 1. 计算每个ITL3区域内每种landuse类型的grid点数量
    landuse_counts = grid_with_regions.groupby(['ITL3', 'landuse']).size()

    # 2. 创建需求映射 (ITL3, landuse) -> demand
    demand_map = {}
    landuse_types = ['residential', 'industrial','commercial', 'agricultural', 'others']
    for _, row in regions.iterrows():
        itl3 = row["ITL3"]
        for landuse_type in landuse_types:
            demand_map[(itl3, landuse_type)] = row[f"{landuse_type}_demand"]

    # 3. 为每个grid点分配其所属landuse类型的需求
    def assign_landuse_demand(row):
        # 使用.get()方法处理一个区域没有某种landuse类型grid的情况
        count = landuse_counts.get((row['ITL3'], row['landuse']), 1)
        total_demand = demand_map.get((row['ITL3'], row['landuse']), 0)
        return total_demand / count if count > 0 else 0

    grid_with_regions['landuse_demand'] = grid_with_regions.apply(assign_landuse_demand, axis=1)

    # =================================================================================
    # 方案二：按【整个ITL3区域】直接均分总需求 (您的新需求)
    # =================================================================================

    # 1. 计算每个ITL3区域的总需求
    demand_cols = [f"{landuse_type}_demand" for landuse_type in landuse_types]
    regions['total_ITL3_demand'] = regions[demand_cols].sum(axis=1)
    # 创建映射： ITL3 -> total_demand
    total_demand_map = regions.set_index('ITL3')['total_ITL3_demand'].to_dict()

    # 2. 计算每个ITL3区域的grid点总数
    # 创建映射： ITL3 -> total_grid_points
    total_grid_count_map = grid_with_regions.groupby('ITL3').size().to_dict()

    # 3. 将总需求均分给区域内所有点 (使用.map()方法，效率更高)
    # 使用 Series.map() 从字典映射值
    total_demand_per_grid = grid_with_regions['ITL3'].map(total_demand_map)
    total_count_per_grid = grid_with_regions['ITL3'].map(total_grid_count_map)

    # 计算平均值并处理可能出现的除零或NaN问题
    grid_with_regions['average_demand'] = (total_demand_per_grid / total_count_per_grid).fillna(0)
    # print(key, 1/total_count_per_grid.unique())

    # --- 数据更新 ---
    # 在最终结果中同时保留两种需求分配方式和原始需求列
    final_columns = [
        'geometry', 'landuse', 'color',
        'landuse_demand', 'average_demand', "Demand (MVA)"
    ]
    # 过滤掉原始数据中不存在的列，以防报错
    final_columns = [col for col in final_columns if col in grid_with_regions.columns]

    interested_locations[key]["grid"] = grid_with_regions[final_columns]

In [22]:
for key in interested_locations.keys():
    for voronoi in ["voronoi", "voronoi_with_ITL3", "clustered_voronoi", "clustered_voronoi_with_ITL3"]:
        voronoi_gdf = interested_locations[key][voronoi][0].copy()
        substations_key = interested_locations[key][voronoi][1].copy()
        grid_gdf = interested_locations[key]["grid"].copy()

        # --- FIX START ---
        # 之前的 .apply 方法效率低且可能因边界问题导致数据丢失，从而造成总需求量不匹配。
        # 使用空间连接（spatial join）能更高效、更准确地完成计算。

        # 1. 使用 sjoin 将网格点连接到其所在的 Voronoi 多边形。
        #    'inner' 确保只保留落在多边形内的点，predicate='within' 定义了空间关系。
        grid_gdf = grid_gdf.to_crs("EPSG:3857")
        voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
        points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
        points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]

        # 2. 按多边形的索引（即 cluster_label）分组，并计算每个多边形内所有点的 'landuse_demand' 之和。
        #    'index_right' 是 sjoin 操作后代表右侧（多边形）DataFrame 索引的列。
        demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['landuse_demand'].sum()

        # 3. 将计算出的总需求映射回 voronoi_gdf。
        #    使用 .reindex 来确保所有多边形都被包含，对于没有包含任何点的多边形，其需求为 0。
        voronoi_gdf['landuse_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
        # --- FIX END ---

        # 分配逻辑保持不变：将每个多边形的总需求均分给其中的所有变电站。
        counts = substations_key["cluster_label"].value_counts()
        substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "landuse_demand"]/counts[x])
        interested_locations[key][voronoi][1] = substations_key
        interested_locations[key][voronoi][0]= voronoi_gdf

In [23]:
grid_gdf

,geometry,landuse,color,landuse_demand,average_demand,Demand (MVA)
0,POINT (-205209.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
1,POINT (-204869.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
2,POINT (-204529.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
3,POINT (-204189.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
4,POINT (-202829.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
...,...,...,...,...,...,...
50117,POINT (-209969.936 7162678.495),others,#b3b3b3ff,0.014052,0.042532,386.696680
50118,POINT (-209629.936 7162678.495),others,#b3b3b3ff,0.014052,0.042532,386.696680
50119,POINT (-209289.936 7162678.495),agricultural,#009682,0.006031,0.042532,386.696680
50120,POINT (-209629.936 7163018.495),agricultural,#009682,0.006031,0.042532,386.696680


In [24]:
for key in interested_locations.keys():
    voronoi_gdf = interested_locations[key]["cluster_voronoi_hdbscan"][0].copy()
    substations_key = interested_locations[key]["cluster_voronoi_hdbscan"][1].copy()
    grid_gdf = interested_locations[key]["grid"].copy()

    # --- FIX START ---
    # 为 civd_gpm 模型应用同样的高效且准确的空间连接聚合方法。
    grid_gdf = grid_gdf.to_crs("EPSG:3857")
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
    points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]
    demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['landuse_demand'].sum()
    voronoi_gdf['landuse_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
    # --- FIX END ---

    counts = substations_key["cluster_label"].value_counts()
    substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "landuse_demand"]/counts[x])
    interested_locations[key]["cluster_voronoi_hdbscan"][1] = substations_key
    interested_locations[key]["cluster_voronoi_hdbscan"][0]= voronoi_gdf

In [27]:
for key in interested_locations.keys():
    voronoi_gdf = interested_locations[key]["cluster_voronoi_hdbscan"][0].copy()
    substations_key = interested_locations[key]["cluster_voronoi_hdbscan"][1].copy()
    grid_gdf = interested_locations[key]["grid"].copy()

    # --- FIX START ---
    # 为 civd_gpm 模型应用同样的高效且准确的空间连接聚合方法。
    grid_gdf = grid_gdf.to_crs("EPSG:3857")
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
    points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]
    demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['average_demand'].sum()
    voronoi_gdf['average_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
    # --- FIX END ---

    counts = substations_key["cluster_label"].value_counts()
    substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "average_demand"]/counts[x])
    interested_locations[key]["cluster_voronoi_hdbscan_avg"] = [voronoi_gdf, substations_key]

In [28]:
grid_gdf

,geometry,landuse,color,landuse_demand,average_demand,Demand (MVA)
0,POINT (-205209.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
1,POINT (-204869.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
2,POINT (-204529.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
3,POINT (-204189.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
4,POINT (-202829.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874
...,...,...,...,...,...,...
50117,POINT (-209969.936 7162678.495),others,#b3b3b3ff,0.014052,0.042532,386.696680
50118,POINT (-209629.936 7162678.495),others,#b3b3b3ff,0.014052,0.042532,386.696680
50119,POINT (-209289.936 7162678.495),agricultural,#009682,0.006031,0.042532,386.696680
50120,POINT (-209629.936 7163018.495),agricultural,#009682,0.006031,0.042532,386.696680


In [29]:
voronoi_gdf

,geometry,landuse_demand,average_demand
cluster_label,,,
0,"MULTIPOLYGON (((-165769.936 7149418.495, -1661...",60.000843,114.005169
1,"POLYGON ((-149109.936 7109978.495, -149109.936...",40.913772,31.077956
2,"POLYGON ((-156929.936 7117458.495, -156929.936...",58.396474,61.571134
3,"POLYGON ((-161349.936 7106578.495, -161349.936...",57.018964,52.837523
4,"MULTIPOLYGON (((-198409.936 7149418.495, -1987...",40.966200,49.921276
...,...,...,...
58,"POLYGON ((-140949.936 7106578.495, -140949.936...",17.869458,16.033850
59,"POLYGON ((-158289.936 7127998.495, -158289.936...",30.369928,49.703473
60,"POLYGON ((-145709.936 7090258.495, -145709.936...",16.632377,18.567594


In [30]:
points_in_voronoi

,geometry,landuse,color,landuse_demand_left,average_demand,Demand (MVA),cluster_label,landuse_demand_right
0,POINT (-205209.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874,35,25.240389
1,POINT (-204869.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874,35,25.240389
2,POINT (-204529.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874,35,25.240389
3,POINT (-204189.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874,35,25.240389
4,POINT (-202829.936 7080058.495),others,#b3b3b3ff,0.007829,0.027999,532.428874,35,25.240389
...,...,...,...,...,...,...,...,...
50117,POINT (-209969.936 7162678.495),others,#b3b3b3ff,0.014052,0.042532,386.696680,22,20.709225
50118,POINT (-209629.936 7162678.495),others,#b3b3b3ff,0.014052,0.042532,386.696680,22,20.709225
50119,POINT (-209289.936 7162678.495),agricultural,#009682,0.006031,0.042532,386.696680,22,20.709225
50120,POINT (-209629.936 7163018.495),agricultural,#009682,0.006031,0.042532,386.696680,22,20.709225


In [31]:
for key in interested_locations.keys():
    voronoi_gdf = interested_locations[key]["voronoi"][0].copy()
    substations_key = interested_locations[key]["voronoi"][1].copy()
    grid_gdf = interested_locations[key]["grid"].copy()

    # --- FIX START ---
    # 对 'average_demand' 应用同样的空间连接修复逻辑，以确保 simple_voronoi 方法的总量守恒。
    grid_gdf = grid_gdf.to_crs("EPSG:3857")
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    points_in_voronoi_with_dups = gpd.sjoin_nearest(grid_gdf, voronoi_gdf, how="inner")
    points_in_voronoi = points_in_voronoi_with_dups[~points_in_voronoi_with_dups.index.duplicated(keep='first')]
    demand_per_polygon = points_in_voronoi.groupby(points_in_voronoi.cluster_label)['average_demand'].sum()
    voronoi_gdf['average_demand'] = demand_per_polygon.reindex(voronoi_gdf.index).fillna(0)
    # --- FIX END ---

    counts = substations_key["cluster_label"].value_counts()
    substations_key["voronoi"] = substations_key["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "average_demand"]/counts[x])
    interested_locations[key]["simple_voronoi"] = [voronoi_gdf, substations_key]

In [32]:
for key in interested_locations.keys():
    # --- Logic for 'average' (ITL3-level) ---
    # 逻辑：在每个 ITL3 区域内，将总需求平均分配给该区域内的所有变电站。
    region_avg_demand = interested_locations[key]["region"].copy()
    substations_key = interested_locations[key]["voronoi"][1].copy()

    # 按 ITL3 计算每个区域的变电站数量
    substation_counts = substations_key.groupby('ITL3').size().reset_index(name='substation_count')

    # 将变电站数量合并回区域需求数据中
    region_avg_demand = region_avg_demand.merge(substation_counts, on='ITL3', how='left')
    region_avg_demand['substation_count'] = region_avg_demand['substation_count'].fillna(1) # 对于没有变电站的区域，填充为1以避免除零错误

    # 计算每个 ITL3 区域的平均需求
    region_avg_demand['avg_demand'] = region_avg_demand['Demand (MVA)'] / region_avg_demand['substation_count']

    # 将计算出的平均需求值合并回主变电站数据中
    substations_key = substations_key.merge(region_avg_demand[['ITL3', 'avg_demand']], on='ITL3', how='left')
    substations_key.rename(columns={'avg_demand': 'ITL3_average'}, inplace=True)

    # --- Start of new 'Global Average' logic ---
    # 逻辑：将整个研究区域视为一个整体，将总需求平均分配给所有变电站。

    # 1. 计算区域内的总需求 (Total Demand)。我们直接对数据中的真实需求求和，这是最准确的总量。
    total_global_demand = substations_key['Demand (MVA)'].sum()
    # 2. 计算区域内的变电站总数
    total_substation_count = len(substations_key)
    # 3. 计算全局平均值并分配给每个变电站
    substations_key['ITL2_average'] = total_global_demand / total_substation_count
    # --- End of 'Global Average' logic ---

    # 将包含所有新计算列的DataFrame存回
    interested_locations[key]["voronoi"][1] = substations_key

In [33]:
# rows = ["simple_voronoi", "voronoi", "voronoi_with_ITL3", "clustered_voronoi", "clustered_voronoi_with_ITL3", "ITL3_average", "ITL2_average", "civd_gpm"]
rows = ["simple_voronoi", "voronoi", "civd", "civd_gpm", "ITL3_average", "ITL2_average"]
cols = interested_locations.keys()
corr_matrix = pd.DataFrame(index=rows, columns=cols)
corr_max_matrix = pd.DataFrame(index=rows, columns=cols)
rmse_matrix = pd.DataFrame(index=rows, columns=cols)
over_matrix = pd.DataFrame(index=rows, columns=cols)
mae_matrix = pd.DataFrame(index=rows, columns=cols)

In [34]:
# for key in ["TLI3", "TLH1"]:
for key in interested_locations.keys():
    substations_key = interested_locations[key]["voronoi"][1].copy()
    substations_key["simple_voronoi"] = interested_locations[key][f"simple_voronoi"][1]["voronoi"].copy()
    substations_key["voronoi_with_ITL3"] = interested_locations[key]["voronoi_with_ITL3"][1]["voronoi"].copy()
    substations_key["clustered_voronoi"] = interested_locations[key]["clustered_voronoi"][1]["voronoi"].copy()
    substations_key["clustered_voronoi_with_ITL3"] = interested_locations[key]["clustered_voronoi_with_ITL3"][1]["voronoi"].copy()
    substations_key[f"civd_gpm"] = interested_locations[key]["cluster_voronoi_hdbscan"][1]["voronoi"].copy()
    substations_key[f"civd"] = interested_locations[key]["cluster_voronoi_hdbscan_avg"][1]["voronoi"].copy()

    for row in rows:
        corr_matrix.loc[row, key] = substations_key["Demand (MVA)"].corr(substations_key[row])
        corr_max_matrix.loc[row, key] = substations_key["Firm Capacity (MVA)"].corr(substations_key[row])
        rmse_matrix.loc[row, key] = np.sqrt(((substations_key["Demand (MVA)"] - substations_key[row]) ** 2).mean())
        over_matrix.loc[row, key] = len(substations_key[substations_key[row] > substations_key["Firm Capacity (MVA)"]])
        mae_matrix.loc[row, key] = np.mean(np.abs(substations_key["Demand (MVA)"] - substations_key[row]))

C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Workspace\Plugins\Miniconda\envs\allocategnn\Lib\site-packages\numpy\lib\_function_base_impl.py:3

In [35]:
substations_key[["Demand (MVA)", "simple_voronoi", "voronoi", "voronoi_with_ITL3", "civd", "civd_gpm", "ITL3_average", "ITL2_average"]].sum()

Demand (MVA)         1928.431438
simple_voronoi       1928.431438
voronoi              1928.431438
voronoi_with_ITL3    1928.431438
civd                 1928.431438
civd_gpm             1928.431438
ITL3_average         1928.431438
ITL2_average         1928.431438
dtype: float64

In [36]:
corr_matrix

,London,TLH1,TLH2,TLH3,TLJ1,TLF1,TLF2,TLC1,TLC2,TLD3,TLD4,TLD6,TLG1,TLG2,TLE3,TLE4
simple_voronoi,0.338269,0.207185,0.022045,0.004171,-0.249542,-0.076141,-0.169463,0.096102,0.2182,0.012673,-0.163372,0.013709,-0.027713,0.304079,-0.195339,-0.197578
voronoi,0.53209,0.525746,0.395107,0.306127,0.211453,0.297182,0.179188,0.168459,0.317137,0.105572,0.269474,0.353825,0.405741,0.705688,-0.079348,0.102119
civd,0.319205,0.13126,-0.091321,-0.024879,-0.185484,-0.086021,-0.097879,-0.017195,-0.080763,0.033332,-0.126817,0.0771,0.014587,0.310807,-0.144927,-0.202281
civd_gpm,0.480846,0.415062,0.231029,0.290125,0.223421,0.33418,0.235814,0.182851,0.341173,0.090907,0.118755,0.464867,0.363484,0.484881,-0.109795,0.005289
ITL3_average,0.604684,0.527017,0.270385,0.269372,0.296196,0.415754,0.370571,0.411866,0.365585,0.213257,0.157578,0.271943,0.41719,0.453414,0.004866,0.185535
ITL2_average,0.0,0.0,NaN,NaN,NaN,-0.0,-0.0,NaN,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,0.0


In [37]:
rmse_matrix

,London,TLH1,TLH2,TLH3,TLJ1,TLF1,TLF2,TLC1,TLC2,TLD3,TLD4,TLD6,TLG1,TLG2,TLE3,TLE4
simple_voronoi,21.085062,7.532179,8.804778,9.268579,15.831154,11.965775,12.573753,18.302037,13.3176,10.199783,11.537243,8.673845,15.492466,13.909478,17.336378,13.769507
voronoi,16.234218,5.513694,6.734706,7.264781,9.981435,8.590706,9.301416,9.964575,9.936022,7.067501,5.636498,6.139346,11.040708,9.563342,9.357358,7.449236
civd,19.932763,7.187897,7.647451,9.068718,13.675693,10.846022,10.883593,17.630256,8.173382,9.591909,9.007742,7.726851,14.763002,12.984063,12.719816,13.353437
civd_gpm,16.219879,5.670413,5.816299,6.727367,8.917241,7.626602,7.541754,8.881564,5.841406,6.578768,4.424193,5.384452,10.738966,11.794767,7.47909,7.007088
ITL3_average,14.429478,5.099184,5.585838,6.661856,7.783992,6.955699,5.917385,5.792556,5.782758,4.382415,3.954248,5.820754,9.878714,11.984424,4.247963,4.588121
ITL2_average,18.116889,6.000063,5.801946,6.917555,8.14969,7.64802,6.370972,6.356752,6.212823,4.485602,4.004275,6.048709,10.869831,13.446006,4.248014,4.669189


In [38]:
mae_matrix

,London,TLH1,TLH2,TLH3,TLJ1,TLF1,TLF2,TLC1,TLC2,TLD3,TLD4,TLD6,TLG1,TLG2,TLE3,TLE4
simple_voronoi,13.583812,6.080047,6.260455,6.66653,11.376924,9.335506,9.944836,10.942943,9.191164,7.160517,8.158716,6.403057,12.481454,9.850878,10.251931,9.988267
voronoi,10.892405,4.3219,5.134696,5.343108,7.49899,6.54051,7.015359,7.300157,7.29311,5.141122,4.481583,4.816737,8.741696,6.424897,6.370239,5.677344
civd,12.885449,5.782599,4.824666,6.172367,9.719898,8.392212,8.461357,9.709582,5.14218,6.545672,5.916204,5.446996,11.444372,9.383586,6.829746,9.620511
civd_gpm,11.026004,4.380494,4.073324,4.525206,6.699354,5.883828,5.904888,6.063807,4.538031,4.632858,3.435417,4.260711,8.56757,7.874055,4.617708,5.346424
ITL3_average,10.185369,3.862863,4.003473,4.148336,6.11781,5.365208,4.868406,4.229629,4.483987,3.209242,3.09208,4.766267,7.613758,8.630405,3.505366,3.441189
ITL2_average,13.535521,4.249879,4.141251,4.251168,6.403693,6.002993,5.101785,4.371362,4.98563,3.273539,3.133389,5.045204,8.535418,9.958762,3.504065,3.459699


In [39]:
with open("./results/intermediate/static_matrix.pickle", "wb") as f:
    pickle.dump({
        "corr_matrix": corr_matrix,
        "rmse_matrix": rmse_matrix,
        "over_matrix": over_matrix,
        "mae_matrix": mae_matrix
    }, f)